# Can the change in Γ's importance values be predicted without naming a cue?

*E35, `experiments/regating_cuefree`, 2026-10-26. Record: `reports/analyses/2026-10-26_analysis_regating-without-cues.md`.
Exploration, provisional.*

## The answer

Partly yes, on these toy tasks. Each held-out input was given the mean change of the eight training inputs whose
original importance values were nearest. On the four tasks with 32 cue words this recovered 0.61 to 0.78 of the
switching effect, against 0.47 to 0.66 for one change per cue word added the same way. 62 to 71% of those nearest inputs
shared the cue, far above the 3% chance with 32 cues but well short of all of them. Restricting them to the same cue helped only where each
cue had its own answer.

## What we were trying to test

**The setup.** Each state is a ten-step Adam fine-tune of all 24 decomposed matrices on a toy task, where a trigger
token is followed by a target. In the *do-nothing* state the update goes into Δ; Γ is frozen but reads the fine-tuned
model, and Δ is left out of the CI-masked model. The *switching effect* is how much the CI-masked model's log-odds of
the target at the trigger rise when Γ reads the fine-tuned model instead of the original. A *recovery* is the share of
that effect a predicted set of importance values gives back. 0 is the original model's values, 1 is each input's own
updated values. All values are at the trigger only.

**The states.** Two have few cue words: " will" followed by " have", one cue, and an emoticon task with ten trigger
tokens. Four have 32 cues: a copy task, a cipher of random word pairs, and cue words for " of", fine-tuned toward their
own answers or all toward " of". Copy and the cipher share one set of inputs and the two " of" states another,
so the four results are two independent input sets, not four.

**The rules.** Every rule gives a held-out input its own original values $g_0$ plus a predicted change, clipped to
$[0, 1]$. The change is fitted on training inputs only, from their change $d$, the fine-tuned values minus the
original ones:

- *One change for every input*, the mean $d$: the original fixed-shift hypothesis.
- *One change per cue word*, the mean $d$ over inputs with the same cue: the current hypothesis, and the reference.
- *Nearest inputs*, the mean $d$ of the eight training inputs with the nearest original values. It names no cue. It
  is not like for like with the per-cue rule: it also lets the change depend on each input's own values. So a match
  does not by itself show that the cue is visible in the original values.
- *Clusters*, the mean $d$ of the input's k-means group, at 8, 32 and 128 groups; random groups are the null.
- *A line per subcomponent*, $a\,g_0 + b$ fitted by least squares: no cue, no other subcomponent.

**One difference from the earlier runs, E32 and E33.** Those runs gave every input of a cue the same values. That averaged form reads
higher, 0.81 to 1.06 per cue, but it erases each input's own values. Here every rule adds a change to the input's own
values, the form a text with no trigger needs at every position.

## What we expected, and why

- **The nearest inputs would match the per-cue rule within 0.1**, medium confidence. At the trigger, Γ's values
  should depend mostly on the trigger token, so the nearest inputs should share the cue.
- **Clusters at 32 groups would match it too**, medium-low: we expected k-means to find roughly the cues.
- **A line per subcomponent would match it**, low confidence: it needs the change to depend only on each
  subcomponent's own original value.
- **On the one-cue task, the cue-free rules would neither gain nor lose** more than 0.1 against one change for every
  input: with one cue there seemed to be nothing to group by.

## What we observed

### Does the change follow the original values, with no cue named?

![arms](../experiments/regating_cuefree/figures/arms.png)

**Reading it.** Yes, against the per-cue rule added the same way. On all four 32-cue tasks the nearest-inputs rule,
green, recovers 0.61 to 0.78. One change per cue word, blue, recovers 0.47 to 0.66, and one change for every input,
grey, 0.01 to 0.13. The nearest-inputs rule falls short of the per-cue rule only on the copy task, by 0.05, and beats
it by 0.31 where all 32 cues lead to " of". Two explanations fit that gain: inputs with different cues sharing
changes, or each input's change depending on its own values, as on the one-cue task below; this run cannot tell them
apart. E32's and E33's per-cue numbers in their averaged form are higher, 0.81 to 1.06, and no rule here reaches them
on the 32-cue tasks.

### Do groups found without labels line up with the cues?

![clusters](../experiments/regating_cuefree/figures/clusters.png)

**Reading it.** No, not at the number of cues. At 32 k-means groups the four 32-cue tasks recover 0.35 to 0.53,
below the per-cue rule's 0.47 to 0.66 on three of the four. A group's most common cue covers only 40 to 57% of its
inputs. With 128 groups recovery reaches 0.49 to 0.70, within 0.06 of the per-cue level or above it on all four. Random
groups of 32 sit at the one-change level, within 0.01, so the gain comes from grouping by original values. The
original values carry the cue, but not as 32 tight clusters.

### Is one fixed line per subcomponent enough?

**Reading it.** Only with few cues. The line per subcomponent, orange in the first figure, recovers 1.02 on the
one-cue task and 0.96 on emoticon, against 0.71 and 0.74 for one change for every input. On the 32-cue tasks it
recovers 0.80 where all cues share " of" and 0.58 on the cipher, but only 0.37 on the copy task and with 32 own
answers. So the change is not one fixed rule per subcomponent in general. When the answers differ, a subcomponent's
change depends on more than its own original value.

### Amendment 1: are the nearest inputs just the ones with the same cue?

An independent check asked whether the nearest inputs simply share the cue. It also asked for the like-for-like
reference: the same nearest rule, with the eight neighbours drawn only from inputs with the same cue.

**Reading it.** Mostly, not wholly. On the 32-cue tasks 71% of the nearest inputs hold the same cue on copy and the
cipher, and 62% on the two " of" tasks, against about 3% by chance; on the few-cue tasks it is 99 to 100%. Drawing the
neighbours from the same cue changes nothing where all cues lead to " of", 0.78 either way, and little on the cipher,
0.68 against 0.69. It adds 0.10 on the copy task and 0.23 where each cue has its own answer, 0.88 against 0.65. So the
original values carry most of what the change depends on. The cue adds information where different cues need
different answers, which the original values at the trigger do not fully separate. No held-out window repeats a
training window, and held-out inputs are as far from their nearest training input as training inputs are from each
other.

The like-for-like comparison also shows how much each input's own values matter within a cue. The nearest rule
inside one cue recovers 0.69 to 0.88, against 0.47 to 0.66 for the cue's mean change added to every input, a gain of
0.05 to 0.33.

## What we learned

The change of Γ's importance values at the trigger can be predicted from the original model's importance values by a
rule that names no cue. Inputs the original Γ treats alike get alike changes. It recovers as much as one change per cue
word added the same way, though less than the averaged per-cue form. The cue adds to the original values only where
cues lead to different answers. Within a cue, the change still depends on each input's own values. The structure is
not a small set of clusters, and a fixed rule per subcomponent is not enough when many answers compete.

## How compelling is the evidence

- **Controls.** All six states were regenerated and reproduced the earlier runs' numbers within 0.0013. A zero change
  recovers exactly 0, and each input's own values recover the reference exactly.
- **Nulls.** Random groups land within 0.01 of one change for every input on every state, over five draws.
- **Coverage.** One fine-tune seed, one 67M model, the trigger position only, six toy tasks, 73 to 192 held-out inputs
  per state.
- **Blind spots.** The distance weighs all alive subcomponents equally, including ones irrelevant to the target. No
  rule's faithfulness under the adversary was checked. Values are written in bf16, so changes smaller than about 0.002
  vanish; this lowers the rules built from small mean changes slightly.

## Surprises

1. **On the one-cue task the cue-free rules beat one change for every input.** We predicted they would match it
   within 0.1. The nearest inputs gave 0.90 against 0.71, and the line per subcomponent 1.02. So with a single cue the
   change still depends on each input's own original values. The earlier finding that inputs' own variation adds only
   a few hundredths was measured in the averaged form, which hides this.
2. **Clusters did not find the cues.** A group's most common cue covered only 40 to 57% of its inputs.
3. **The nearest inputs often carry a different cue.** We expected at least 90% to share it; 62 to 71% did.

## What next

1. **E36, already designed:** the same rules at every position of a code-text fine-tune with no trigger. If the
   nearest-inputs rule recovers most of the switching there, the restated hypothesis carries to text with no cue.
2. **The line's slopes:** whether the varying subcomponents have slopes below 1, so that each is pulled toward a set
   value. E36 records this.

## Scorecard

Filed 2026-10-26 16:48 (`experiments/regating_cuefree/results/predictions.json`, sha256 `2e3df9b2…`), before the run.
The filed decision tree had no branch for the nearest rule holding while the clusters failed; making the nearest rule
E36's headline test was decided after the run.

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| G1 | emoticon reproduces E32; zero change recovers 0 | r, net forms, filed arm, zero checks | all differences 0.0 | pass |
| G2 | the other states reproduce E32/E33 | r ±0.05, net forms ±0.02 | largest difference 0.0013 | pass on all five |
| C_one | one cue: per cue equals one change for every input | difference ≤ 0.001 | 0 | pass |
| C | one cue: nearest and 8 clusters within 0.1 of one change | both within 0.1 | 0.90, 0.79 vs 0.71 | **fail, upward** |
| N | random groups within 0.1 of one change | every state | within 0.01 everywhere | pass |
| F | nearest within 0.1 of per cue on each 32-cue state | R_near ≥ R_cue − 0.1 | −0.05, +0.05, +0.10, +0.31 | pass |
| K | 32 clusters within 0.1 of per cue | R_clu_32 ≥ R_cue − 0.1 | −0.23, −0.18, −0.20, +0.07 | **fail** on three of four |
| A | a line per subcomponent matches per cue and beats one change by 0.2 | both | pass on cipher and " of", fail on copy and own answers | **fail** |

Amendment 1, filed 2026-10-26 17:19 (`results/predictions_amend1.json`, sha256 `dff07fe0…`), before its run.

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| A0 | the nearest rule reproduces E35 | within 0.001 on every state | exact on all six | pass |
| L | crossing cue boundaries costs at most 0.1 | R_near ≥ R_near_cue − 0.1 on each 32-cue state | −0.10, −0.02, −0.23, −0.01 | **fail** on own answers |
| T | the nearest inputs share the cue | share ≥ 0.9 on each 32-cue state | 0.71, 0.71, 0.62, 0.62 | **fail** |
| D | held-out windows are not training windows | ≤ 5% duplicated | 0 on every state | pass |

## Not run

Seeds beyond 0; positions other than the trigger; the nearest rule in the averaged form; the slopes of the varying
subcomponents at the trigger; any distance other than Euclidean over all alive subcomponents; faithfulness under the
adversary of any rule; SGD or small-update states; a larger model.

## Provenance

- Results `experiments/regating_cuefree/results/regating_cuefree.json` (sha256 `874675d3…`), log
  `results/large/full.log` (sha256 `ece43666…`). RunPod RTX 4090, pod `01M5WHFHRXXYVRJG3AWGPNH330`, 2026-10-26
  16:42–16:51; run 58 s; $0.13, plus $0.06 for a first pod that failed in set-up.
- Amendment 1: `results/regating_cuefree_amend1.json` (sha256 `52ca0cb3…`), log `results/large/amend1.log`; RunPod
  4090 pod `01M5WK51FH0VXCAD1HRZKCT61T`, 17:19–17:21, shared with E36.
- Code `experiments/regating_cuefree/run.py`, shared steps in `src/commute/cuefree.py`; plan
  `reports/plans/2026-10-26_plan_regating-without-cues.md`; figures `experiments/regating_cuefree/figures/{arms,clusters}.png`,
  from `plot.py`.